# Laboratorio 06.2 — Evaluación de recuperación

Evaluaremos rankings de documentos con relevancia conocida y compararemos una línea base lexical con recuperación vectorial TF-IDF.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Preparar consultas y documentos relevantes como juicios de relevancia.
- Calcular Hit Rate@K, Precision@K, Recall@K y MRR.
- Comparar estrategias con el mismo corpus y conjunto de consultas.
- Diagnosticar errores de recuperación antes de evaluar generación.

## 2. Conceptos

Hit Rate@K mide si aparece algún elemento relevante; Precision@K mide qué fracción de los primeros K lo es; Recall@K mide qué proporción de todos los relevantes se recupera. MRR pondera la posición del primer resultado relevante. Ninguna métrica es suficiente por sí sola: define unidad documental, relevancia y K de acuerdo con el uso downstream.

## 3. Arquitectura

```mermaid
flowchart LR
    A[Consultas de prueba] --> C[Ranker lexical]
    B[Juicios de relevancia] --> D[Metricas top-k]
    A --> E[Ranker TF-IDF]
    C --> D
    E --> D
    D --> F[Analisis por consulta y estrategia]
```

## 4. Preparación del entorno

Cargamos el corpus y las métricas reutilizables. La búsqueda y evaluación son locales; no se generan respuestas ni se usan APIs.

In [ ]:
import re

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from llm_engineering.environment import preparar_entorno
from llm_engineering.evaluation.metrics import (
    hit_rate_at_k, mean_reciprocal_rank, precision_at_k, recall_at_k,
)
from llm_engineering.retrieval.chunking import chunk_text
from llm_engineering.retrieval.vector_search import rank_by_cosine

raiz_curso = preparar_entorno()
documentos = pd.read_json(raiz_curso / 'data' / 'documents' / 'faq.jsonl', lines=True).to_dict(orient='records')

## 5. Definición del problema

Recuperamos evidencia para tres preguntas de soporte. Cada consulta tiene un conjunto de IDs documentales relevantes definido antes de ejecutar los rankers. Fragmentamos todos los documentos igual para no confundir el efecto del chunking con el del método de búsqueda.

## 6. Línea base

Un ranker lexical puntúa el número de términos compartidos y mantiene un orden estable en empates. Es interpretable y sirve como comparación, pero no modela importancia por frecuencia ni paráfrasis.

In [ ]:
consultas = [
    {'id': 'q_envio', 'texto': 'seguimiento paquete retraso entrega', 'relevantes': {'envio_seguimiento', 'envio_entrega'}},
    {'id': 'q_pago', 'texto': 'cargo duplicado movimiento tarjeta', 'relevantes': {'pago_duplicado'}},
    {'id': 'q_devolucion', 'texto': 'devolucion 30 dias entrega producto', 'relevantes': {'devolucion_plazo'}},
]
tamano_fragmento = 12
solapamiento = 2
fragmentos = [
    {**fragmento, 'documento_id': documento['id'], 'tema': documento['tema']}
    for documento in documentos
    for fragmento in [
        {'indice': indice, 'texto': texto}
        for indice, texto in enumerate(chunk_text(documento['texto'], tamano_fragmento, solapamiento))
    ]
]

def rankear_lexical(consulta: str, k: int) -> list[int]:
    """Ordena por terminos compartidos y conserva solo un fragmento por documento."""
    terminos = set(re.findall(r'\w+', consulta.casefold(), flags=re.UNICODE))
    puntuaciones = [
        len(terminos & set(re.findall(r'\w+', fragmento['texto'].casefold(), flags=re.UNICODE)))
        for fragmento in fragmentos
    ]
    ordenados = sorted(range(len(fragmentos)), key=lambda indice: (-puntuaciones[indice], indice))
    indices_unicos = []
    documentos_vistos = set()
    for indice in ordenados:
        documento_id = fragmentos[indice]['documento_id']
        if documento_id not in documentos_vistos:
            documentos_vistos.add(documento_id)
            indices_unicos.append(indice)
        if len(indices_unicos) == k:
            break
    return indices_unicos

print(f'Consultas: {len(consultas)}; fragmentos indexados: {len(fragmentos)}')

## 7. Implementación

Ajustamos un vectorizador a todos los fragmentos y lo usamos para representar cada consulta. Los dos métodos comparten los mismos fragmentos y el mismo K para que la comparación sea interpretable.

In [ ]:
vectorizador = TfidfVectorizer()
matriz = vectorizador.fit_transform([fragmento['texto'] for fragmento in fragmentos]).toarray()

def rankear_tfidf(consulta: str, k: int) -> list[int]:
    """Ordena por cosine TF-IDF y conserva un solo fragmento por documento."""
    vector = vectorizador.transform([consulta]).toarray()[0]
    ranking_completo = rank_by_cosine(vector, matriz, top_k=len(fragmentos))
    indices_unicos = []
    documentos_vistos = set()
    for indice, _ in ranking_completo:
        documento_id = fragmentos[indice]['documento_id']
        if documento_id not in documentos_vistos:
            documentos_vistos.add(documento_id)
            indices_unicos.append(indice)
        if len(indices_unicos) == k:
            break
    return indices_unicos

print('Forma de la matriz de fragmentos:', matriz.shape)

## 8. Experimentos

Evaluamos ambos rankers con K=3 para cada consulta y luego variamos K en TF-IDF. La relevancia no cambia entre corridas; las métricas se calculan a nivel de documento padre para no contar fragmentos repetidos como documentos distintos.

In [ ]:
filas = []
for consulta in consultas:
    for estrategia, funcion in (('lexical', rankear_lexical), ('tfidf', rankear_tfidf)):
        indices = funcion(consulta['texto'], 3)
        ids = [fragmentos[indice]['documento_id'] for indice in indices]
        filas.append({
            'consulta_id': consulta['id'],
            'estrategia': estrategia,
            'k': 3,
            'hit_rate': hit_rate_at_k(ids, consulta['relevantes'], 3),
            'precision_at_k': precision_at_k(ids, consulta['relevantes'], 3),
            'recall_at_k': recall_at_k(ids, consulta['relevantes'], 3),
            'ranking_documentos': ids,
        })
resultados = pd.DataFrame(filas)
display(resultados)
resultados.to_csv(raiz_curso / 'outputs' / '06_02_retrieval_by_query.csv', index=False)

## 9. Evaluación

Calculamos MRR por estrategia sobre el ranking completo, además de promedios para las métricas@3. MRR premia encontrar pronto el primer documento relevante, mientras Recall considera todos los relevantes.

In [ ]:
filas_mrr = []
for estrategia, funcion in (('lexical', rankear_lexical), ('tfidf', rankear_tfidf)):
    rankings = []
    relevantes = []
    for consulta in consultas:
        indices = funcion(consulta['texto'], len(fragmentos))
        rankings.append([fragmentos[indice]['documento_id'] for indice in indices])
        relevantes.append(consulta['relevantes'])
    filas_mrr.append({'estrategia': estrategia, 'mrr': mean_reciprocal_rank(rankings, relevantes)})
resumen = resultados.groupby('estrategia', as_index=False).agg(
    hit_rate_at_3=('hit_rate', 'mean'),
    precision_at_3=('precision_at_k', 'mean'),
    recall_at_3=('recall_at_k', 'mean'),
)
resumen = resumen.merge(pd.DataFrame(filas_mrr), on='estrategia')
display(resumen)
resumen.to_csv(raiz_curso / 'outputs' / '06_02_retrieval_summary.csv', index=False)

## 10. Discusión

Los resultados dependen de juicios de relevancia, chunk size, solapamiento y consultas. Un ranker puede tener buen Hit Rate y baja Precision si añade documentos irrelevantes; una recuperación débil no se corrige confiablemente con un generador más potente. Revisa ejemplos fallidos además del promedio.

## 11. Ejercicios

1. Añade un segundo documento relevante a la consulta de pagos.
2. Compara el efecto de K=1, 3 y 5 en Precision y Recall.
3. Crea un caso sin documentos relevantes y define cómo evaluar abstención.
4. Añade una consulta con sinónimos y explica el fallo lexical.

## 12. Desafío

Construye un conjunto etiquetado más amplio y compara TF-IDF con un embedding denso. Usa separación de consultas de desarrollo y prueba para evitar ajustar el ranker a las respuestas evaluadas.

## 13. Ideas clave

- Evalúa retrieval con consultas y relevancia conocidas.
- Hit Rate, Precision, Recall y MRR miden propiedades diferentes.
- Mantén chunking fijo al comparar rankers.
- Recuperación es un componente evaluable por separado de generación.